<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/17_6_1_THEORY_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#LangChain

This notebook demonstrates how to build a sentiment analysis and review summarization system using LangChain, OpenAI’s API, and various NLP libraries. The system processes customer reviews of McDonald's, specifically focusing on extracting insights and ratings based on different aspects of the service (e.g., customer service, food quality). It first installs necessary libraries, loads a dataset of McDonald's reviews, and then creates text embeddings to facilitate similarity searches. The notebook then sets up a language model to interact with these embeddings, allowing users to query specific aspects of the service.

## install packages

In [ ]:
!pip install langchain > /dev/null
!pip install sentence_transformers > /dev/null
!pip install chromadb > /dev/null
!pip install openai > /dev/null

In [ ]:
!pip install langchain_community -q
!pip install langchain_openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 1.9 MB/s eta 0:00:00


## add API key

In [ ]:
import getpass

openai_api_key = getpass.getpass(prompt='Enter your OpenAI API Key: ')

Enter your OpenAI API Key: ··········


In [ ]:
!wget https://raw.githubusercontent.com/Gaalipour/Opinion-Mining-in-Costumer-Reviews-for-McDonalds-Restaurants/master/McDonalds-Yelp-Sentiment-DFE.csv

--2025-07-10 23:24:28--  https://raw.githubusercontent.com/Gaalipour/Opinion-Mining-in-Costumer-Reviews-for-McDonalds-Restaurants/master/McDonalds-Yelp-Sentiment-DFE.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 912493 (891K) [application/octet-stream]
Saving to: ‘McDonalds-Yelp-Sentiment-DFE.csv’

McDonalds-Yelp-Sent 100%[===================>] 891.11K  --.-KB/s    in 0.06s   

2025-07-10 23:24:28 (15.1 MB/s) - ‘McDonalds-Yelp-Sentiment-DFE.csv’ saved [912493/912493]



## load dataset

In [ ]:
import pandas as pd

NUM_REVIEWS = 20

dataset = pd.read_csv("McDonalds-Yelp-Sentiment-DFE.csv", encoding="ISO-8859-1")["review"].head(NUM_REVIEWS)
dataset.to_csv("reviews.csv", encoding="ISO-8859-1", index=False)
dataset.head(10)

,review
0,"I'm not a huge mcds lover, but I've been to be..."
1,Terrible customer service. Î¾I came in at 9:30...
2,"First they ""lost"" my order, actually they gave..."
3,I see I'm not the only one giving 1 star. Only...
4,"Well, it's McDonald's, so you know what the fo..."
5,This has to be one of the worst and slowest Mc...
6,I'm not crazy about this McDonald's. Î¾This is...
7,One Star and I'm beng kind. I blame management...
8,Never been upset about any fast food drive thr...
9,This McDonald's has gotten much better. Usuall...


## create embeddings and store them

In [ ]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma


loader = CSVLoader(file_path="./reviews.csv")
data = loader.load()

print("Creating embeddings...")
embeddings = SentenceTransformerEmbeddings()

docsearch = Chroma.from_documents(
    data, embeddings
)


Creating embeddings...


/tmp/ipython-input-6-2754156822.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings()
/tmp/ipython-input-6-2754156822.py:10: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = SentenceTransformerEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## create search tool

In [ ]:
from langchain.agents import Tool

def search(search_input):
    docs = docsearch.similarity_search_with_score(search_input, k=2)
    return docs


tools = [
    Tool(
        name="Search",
        func=search,
        description="return reviews about mc donalds food related to the search term",
    )
]

## get llm, which will decide the use of tool

In [ ]:
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.0, openai_api_key=openai_api_key)

In [ ]:
from langchain.prompts import PromptTemplate


template = "How is the quality of {service} at McDonalds according to customer reviews? Highlight positive and negative comments if any and set a rating between 1 to 5 stars"
prompt = PromptTemplate(
    input_variables=["service"],
    template=template
)

## create agent with llm and tool

In [ ]:
from langchain.agents import AgentType, initialize_agent


agent = initialize_agent(
    tools,
    llm = llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

/tmp/ipython-input-11-2341326071.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [ ]:
response = agent.run(prompt.format_prompt(service="potatoes"))

/tmp/ipython-input-12-4222717931.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run(prompt.format_prompt(service="potatoes"))




> Entering new AgentExecutor chain...
I should search for reviews about McDonald's potatoes to find out the quality according to customer feedback.
Action: Search
Action Input: "McDonald's potatoes quality reviews"
Observation: [(Document(metadata={'source': './reviews.csv', 'row': 19}, page_content='review: This is the worst McDonalds I have ever seen. ξThe service was terrible and the employees were either lazy or incompetent. I will never return to this establishment.'), 0.8473734855651855), (Document(metadata={'row': 5, 'source': './reviews.csv'}, page_content="review: This has to be one of the worst and slowest McDonald's franchises there is. Can't figure out why my Egg McMuffin is always on a stale un-toasted English muffin. Bought A chocolate shake today and threw it away."), 0.8721626996994019)]
Thought:I need to look for more reviews to get a better understanding of the quality of McDonald's potatoes.
Action: Search
Action Input: "McDonald's potatoes quality reviews"
Observa

## create output format chain

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

class Review(BaseModel):
    positive: str = Field(title="positive", description="list of positive things mentioned by customer reviews, if any")
    negative: str = Field(title="negative", description="list of negative things mentioned by customer reviews, if any")
    rating: int = Field(title="rating", description="numeric value between 1 and 5 that represents customer rating")

output_parser = JsonOutputParser(pydantic_object=Review)

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
output_parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"positive": {"title": "positive", "description": "list of positive things mentioned by customer reviews, if any", "type": "string"}, "negative": {"title": "negative", "description": "list of negative things mentioned by customer reviews, if any", "type": "string"}, "rating": {"title": "rating", "description": "numeric value between 1 and 5 that represents customer rating", "type": "integer"}}, "required": ["positive", "negative", "rating"]}\n```'

In [ ]:
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain

format_instructions = output_parser.get_format_instructions()
template = "{review}\n{format_instructions}"
prompt_template = PromptTemplate(
    input_variables=["review"],
    template=template,
    partial_variables={"format_instructions": format_instructions}
)
format_chain = LLMChain(llm=llm, prompt=prompt_template)

/tmp/ipython-input-15-575725674.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  format_chain = LLMChain(llm=llm, prompt=prompt_template)


## create final sequential chain with agent + format & Run!

In [ ]:
overall_chain = SimpleSequentialChain(chains=[agent, format_chain], verbose=True)

In [ ]:
result = overall_chain.run("food")



> Entering new SimpleSequentialChain chain...


> Entering new AgentExecutor chain...
you should search for reviews related to McDonald's food
Action: Search
Action Input: "McDonald's food reviews"
Observation: [(Document(metadata={'source': './reviews.csv', 'row': 19}, page_content='review: This is the worst McDonalds I have ever seen. ξThe service was terrible and the employees were either lazy or incompetent. I will never return to this establishment.'), 0.43290048837661743), (Document(metadata={'row': 9, 'source': './reviews.csv'}, page_content="review: This McDonald's has gotten much better. Usually my order would be wrong every single time so I would not leave that window until I checked every single item. I only hit up fast food once a month or so and it needs to be worth it. Also the fries used to be cold and the cheese on the burger was never melted. Everything was just lukewarm. Now my order has been right a few times in a row and my food hot. Also, I love dining room. Usua

In [ ]:
output_parser.parse(result)

{'positive': 'The food was fresh and hot.',
 'negative': 'The fries were soggy and the burger was cold.',
 'rating': 2}

In [ ]:
result = overall_chain.run("food")



> Entering new SimpleSequentialChain chain...


> Entering new AgentExecutor chain...
I should search for reviews about McDonald's food to gather information.
Action: Search
Action Input: "McDonald's food reviews"
Observation: [(Document(metadata={'source': './reviews.csv', 'row': 19}, page_content='review: This is the worst McDonalds I have ever seen. ξThe service was terrible and the employees were either lazy or incompetent. I will never return to this establishment.'), 0.43290048837661743), (Document(metadata={'row': 9, 'source': './reviews.csv'}, page_content="review: This McDonald's has gotten much better. Usually my order would be wrong every single time so I would not leave that window until I checked every single item. I only hit up fast food once a month or so and it needs to be worth it. Also the fries used to be cold and the cheese on the burger was never melted. Everything was just lukewarm. Now my order has been right a few times in a row and my food hot. Also, I love d